# Reference-Metric Applications

## Author: Zach Etienne

To run the whole notebook, click the `>>` toolbar button and choose
**Restart Kernel and Run All Cells...**.

This notebook inspects reference metrics as reusable coordinate geometry for
curvilinear generated code, then validates determinant identities and BHaH
precompute metadata.

Navigation: [Index][index] | Previous: [Cartesian Wave Project][prev]
| Next: [Basis Transforms][next]

[index]: ../index.ipynb
[prev]: ../3-wave_equation/start_to_finish_wave_cartesian.ipynb
[next]: basis_transforms.ipynb

### Required Reading

- [Reference Metrics](../1-intro/reference_metric.ipynb)
- [Start-to-Finish Cartesian Wave Project][cart-wave]

[cart-wave]: ../3-wave_equation/start_to_finish_wave_cartesian.ipynb

### NRPy Source Code

- [Reference metrics](https://github.com/nrpy/nrpy/blob/main/nrpy/reference_metric.py): defines coordinate
  maps, scale factors, metric components, and determinant expressions.
- [BHaH reference-metric precompute][precompute-source]: defines the reusable
  coordinate-only member list used by generated BHaH loops.

[precompute-source]: https://github.com/nrpy/nrpy/blob/main/nrpy/infrastructures/BHaH/rfm_precompute.py

# Table of Contents

1. [Words for This Notebook](#Words-for-This-Notebook)
1. [Notation and Terms](#Notation-and-Terms)
1. [Step 1](#Step-1:-Import-Tools): Import reference-metric tools.
1. [Step 2](#Step-2:-Inspect-Geometry-Data): Inspect geometry catalogs.
1. [Validation Check](#Validation-Check): Validate determinant identities.
1. [Step 3](#Step-3:-Inspect-Precompute-Members): Inspect precompute members.
1. [Precompute Validation](#Precompute-Validation): Validate member names.
1. [Learning Check](#Learning-Check)
1. [Continue](#Continue)

# Words for This Notebook
### [Back to [top](#Table-of-Contents)]

- **Reference metric:** the built-in geometry of a coordinate system.
- **Coordinate map:** the formulas that map a coordinate system to Cartesian
  coordinates.
- **Scale factor:** a coordinate-dependent length factor for one coordinate
  direction.
- **Determinant:** a scalar built from a matrix. For a metric it measures
  coordinate-volume scaling.
- **Orthogonal coordinates:** coordinates whose basis directions meet at
  right angles, so the reference metric has no off-diagonal entries.
- **Coordinate singularity:** a point or line where coordinate labels behave
  badly even when the modeled space is regular.
- **BHaH:** the BlackHoles@Home infrastructure target used by generated NRPy
  C projects.
- **CodeParameter:** a runtime setting that generated C code reads from
  project parameter structures.
- **Generated loop:** C code that visits grid points and evaluates formulas.
- **Coordinate-only expression:** a formula depending only on coordinates and
  coordinate parameters, not on evolved fields.
- **Precompute member:** a named coordinate-only expression or derivative that
  generated loops can store and reuse.

# Notation and Terms
### [Back to [top](#Table-of-Contents)]

This notebook uses three-dimensional coordinate systems. The coordinate
variables are stored as `xx[0]`, `xx[1]`, and `xx[2]`.

In orthogonal coordinates, the reference metric diagonal is built from scale
factors:

$$
\hat{\gamma}_{ii} = h_i^2.
$$

The determinant identity checked below is

$$
\det(\hat{\gamma}) = (h_0 h_1 h_2)^2.
$$

The catalog compares four named reference metrics. Each row names the
coordinate system, states why it matters, and identifies the output to inspect.

| Coordinate system | Purpose | Where used | What to inspect |
| --- | --- | --- | --- |
| `Cartesian` | baseline with no stretching | comparison case | unit factors |
| `Spherical` | radius and angle volume factors | curvilinear waves | `r`, `sin` |
| `SinhSpherical` | concentrated spherical radius | stretched grids | sinh factors |
| `SinhCylindrical` | stretched radius and height | precompute example | members |

# Step 1: Import Tools
### [Back to [top](#Table-of-Contents)]

This setup cell imports SymPy, the NRPy parameter interface, reference metrics,
and the BHaH precompute helper. It has no output because the next cell prints
the geometry catalog.

In [1]:
import sympy as sp

import nrpy.params as par
import nrpy.reference_metric as refmetric
from nrpy.infrastructures.BHaH.rfm_precompute import ReferenceMetricPrecompute

# Step 2: Inspect Geometry Data
### [Back to [top](#Table-of-Contents)]

A curvilinear generated RHS needs coordinate maps, scale factors, metric
determinants, and radial-like directions before it can write geometry-aware
terms.

Inspect each coordinate-system block for:

- coordinate variables `xx`;
- scale factors;
- determinant `detgammahat`;
- radial-like directions;
- Cartesian coordinate map;
- diagonal entries of `ghatDD` and `ghatUU`.

In [2]:
coord_systems = ["Cartesian", "Spherical", "SinhSpherical", "SinhCylindrical"]
catalog_rows = []
print("coord_system | purpose | inspect")
for coord_system in coord_systems:
    purpose = {
        "Cartesian": "baseline with no stretching",
        "Spherical": "radius and angle volume factors",
        "SinhSpherical": "concentrated spherical radius",
        "SinhCylindrical": "stretched radius and height",
    }[coord_system]
    print(coord_system, "|", purpose, "|", "map, factors, determinant")
print()

for coord_system in coord_systems:
    metric = refmetric.ReferenceMetric(coord_system, enable_rfm_precompute=False)
    catalog_rows.append((coord_system, metric))
    print("coordinate system:", coord_system)
    print("  xx:", metric.xx)
    print("  scale factors:", metric.scalefactor_orthog)
    print("  detgammahat:", metric.detgammahat)
    print("  radial_like_dirs:", metric.radial_like_dirns)
    print("  Cartesian map:", metric.xx_to_Cart)
    print("  ghatDD diagonal:", [metric.ghatDD[i][i] for i in range(3)])
    print("  ghatUU diagonal:", [metric.ghatUU[i][i] for i in range(3)])

coord_system | purpose | inspect
Cartesian | baseline with no stretching | map, factors, determinant
Spherical | radius and angle volume factors | map, factors, determinant
SinhSpherical | concentrated spherical radius | map, factors, determinant
SinhCylindrical | stretched radius and height | map, factors, determinant

coordinate system: Cartesian
  xx: [xx0, xx1, xx2]
  scale factors: [1, 1, 1]
  detgammahat: 1
  radial_like_dirs: [0, 1, 2]
  Cartesian map: [xx0, xx1, xx2]
  ghatDD diagonal: [1, 1, 1]
  ghatUU diagonal: [1, 1, 1]
coordinate system: Spherical
  xx: [xx0, xx1, xx2]
  scale factors: [1, xx0, xx0*sin(xx1)]
  detgammahat: xx0**4*sin(xx1)**2
  radial_like_dirs: [0]
  Cartesian map: [xx0*sin(xx1)*cos(xx2), xx0*sin(xx1)*sin(xx2), xx0*cos(xx1)]
  ghatDD diagonal: [1, xx0**2, xx0**2*sin(xx1)**2]
  ghatUU diagonal: [1, xx0**(-2), 1/(xx0**2*sin(xx1)**2)]
coordinate system: SinhSpherical
  xx: [xx0, xx1, xx2]
  scale factors: [AMPL*(exp(xx0/SINHW)/SINHW + exp(-xx0/SINHW)/SINHW)/(

coordinate system: SinhCylindrical
  xx: [xx0, xx1, xx2]
  scale factors: [AMPLRHO*(exp(xx0/SINHWRHO)/SINHWRHO + exp(-xx0/SINHWRHO)/SINHWRHO)/(exp(1/SINHWRHO) - exp(-1/SINHWRHO)), AMPLRHO*(exp(xx0/SINHWRHO) - exp(-xx0/SINHWRHO))/(exp(1/SINHWRHO) - exp(-1/SINHWRHO)), AMPLZ*(exp(xx2/SINHWZ)/SINHWZ + exp(-xx2/SINHWZ)/SINHWZ)/(exp(1/SINHWZ) - exp(-1/SINHWZ))]
  detgammahat: AMPLRHO**4*AMPLZ**2*(exp(xx0/SINHWRHO)/SINHWRHO + exp(-xx0/SINHWRHO)/SINHWRHO)**2*(exp(xx2/SINHWZ)/SINHWZ + exp(-xx2/SINHWZ)/SINHWZ)**2*(exp(xx0/SINHWRHO) - exp(-xx0/SINHWRHO))**2/((exp(1/SINHWRHO) - exp(-1/SINHWRHO))**4*(exp(1/SINHWZ) - exp(-1/SINHWZ))**2)
  radial_like_dirs: [0, 2]
  Cartesian map: [AMPLRHO*(exp(xx0/SINHWRHO) - exp(-xx0/SINHWRHO))*cos(xx1)/(exp(1/SINHWRHO) - exp(-1/SINHWRHO)), AMPLRHO*(exp(xx0/SINHWRHO) - exp(-xx0/SINHWRHO))*sin(xx1)/(exp(1/SINHWRHO) - exp(-1/SINHWRHO)), AMPLZ*(exp(xx2/SINHWZ) - exp(-xx2/SINHWZ))/(exp(1/SINHWZ) - exp(-1/SINHWZ))]
  ghatDD diagonal: [AMPLRHO**2*(exp(xx0/SINHWRHO)/SINHW

# Validation Check
### [Back to [top](#Table-of-Contents)]

The trusted result is the determinant identity
`det(gammahat) = (h_0 h_1 h_2)^2` for each orthogonal reference metric. The
newly computed result is each determinant expression stored by NRPy.

In [3]:
determinant_residuals = {}
for coord_system, metric in catalog_rows:
    scale_product = sp.prod(metric.scalefactor_orthog)
    residual = sp.trigsimp(sp.simplify(metric.detgammahat - scale_product**2))
    determinant_residuals[coord_system] = residual

print("coord_system | determinant residual")
for coord_system, residual in determinant_residuals.items():
    print(coord_system, "|", residual)

failed_residuals = {
    coord_system: residual
    for coord_system, residual in determinant_residuals.items()
    if residual != 0
}
if failed_residuals:
    raise RuntimeError(f"Nonzero determinant residuals: {failed_residuals}")
print("validation status: determinant identities passed")

coord_system | determinant residual
Cartesian | 0
Spherical | 0
SinhSpherical | 0
SinhCylindrical | 0
validation status: determinant identities passed


# Step 3: Inspect Precompute Members
### [Back to [top](#Table-of-Contents)]

Generated RHS loops may need the same coordinate-only functions at every grid
point. BHaH precompute support stores those functions and derivatives once,
then passes the stored arrays through generated data structures.

Inspect the member names for:

- `f0_of_xx0`, a reusable function of `xx[0]`;
- `__D0`, the first derivative with respect to `xx[0]`;
- `__DD00`, the second derivative with respect to `xx[0]`;
- `f3_of_xx2`, a reusable function of `xx[2]`;
- allocation sizes tied to one coordinate direction.

This cell changes runtime settings only long enough to print the member list,
then restores the original values.

In [4]:
saved_runtime_settings = {
    name: par.parval_from_str(name)
    for name in [
        "parallelization",
        "fp_type",
        "CoordSystem_to_register_CodeParameters",
    ]
}
try:
    par.set_parval_from_str("parallelization", "openmp")
    par.set_parval_from_str("fp_type", "double")
    par.set_parval_from_str(
        "CoordSystem_to_register_CodeParameters", "SinhCylindrical"
    )
    precompute_metric = refmetric.reference_metric["SinhCylindrical_rfm_precompute"]
    precompute = ReferenceMetricPrecompute("SinhCylindrical")
    member_names = [name for name, _ in precompute.member_specs]
    print("precompute metric:", precompute_metric.CoordSystem)
    print("stored member count:", len(precompute.member_specs))
    print("member | allocation")
    for name, declaration in precompute.member_specs:
        print(name, "|", declaration)
finally:
    for name, value in saved_runtime_settings.items():
        par.set_parval_from_str(name, value)
print("restored runtime settings:", sorted(saved_runtime_settings))

Setting up reference_metric[SinhCylindrical_rfm_precompute]...
precompute metric: SinhCylindrical
stored member count: 7
member | allocation
f0_of_xx0 | (size_t)params->Nxx_plus_2NGHOSTS0
f0_of_xx0__D0 | (size_t)params->Nxx_plus_2NGHOSTS0
f0_of_xx0__DD00 | (size_t)params->Nxx_plus_2NGHOSTS0
f0_of_xx0__DDD000 | (size_t)params->Nxx_plus_2NGHOSTS0
f3_of_xx2 | (size_t)params->Nxx_plus_2NGHOSTS2
f3_of_xx2__D2 | (size_t)params->Nxx_plus_2NGHOSTS2
f3_of_xx2__DD22 | (size_t)params->Nxx_plus_2NGHOSTS2
restored runtime settings: ['CoordSystem_to_register_CodeParameters', 'fp_type', 'parallelization']


# Precompute Validation
### [Back to [top](#Table-of-Contents)]

The trusted result is the complete expected member-name set for the
`SinhCylindrical` precompute example. The newly computed result is the member
list returned by `ReferenceMetricPrecompute`.

In [5]:
expected_members = {
    "f0_of_xx0",
    "f0_of_xx0__D0",
    "f0_of_xx0__DD00",
    "f0_of_xx0__DDD000",
    "f3_of_xx2",
    "f3_of_xx2__D2",
    "f3_of_xx2__DD22",
}
actual_members = set(member_names)
missing_members = sorted(expected_members - actual_members)
extra_members = sorted(actual_members - expected_members)
print("expected members:", sorted(expected_members))
print("actual members:", sorted(actual_members))
print("missing members:", missing_members)
print("extra members:", extra_members)
if missing_members or extra_members:
    raise RuntimeError("Unexpected SinhCylindrical precompute member set.")
print("validation status: precompute member set passed")

expected members: ['f0_of_xx0', 'f0_of_xx0__D0', 'f0_of_xx0__DD00', 'f0_of_xx0__DDD000', 'f3_of_xx2', 'f3_of_xx2__D2', 'f3_of_xx2__DD22']
actual members: ['f0_of_xx0', 'f0_of_xx0__D0', 'f0_of_xx0__DD00', 'f0_of_xx0__DDD000', 'f3_of_xx2', 'f3_of_xx2__D2', 'f3_of_xx2__DD22']
missing members: []
extra members: []
validation status: precompute member set passed


The coordinate catalogs show the geometry data that curvilinear generated code
needs. The determinant residuals validate the metric-scale-factor identity.
The precompute member check validates the named coordinate-only arrays that
BHaH generated loops can reuse.

# Learning Check
### [Back to [top](#Table-of-Contents)]

Use the `Spherical` row and determinant residual table to identify which
scale factors create the `r` and `sin(theta)` volume factor. Then use the
precompute member table to explain why a generated loop should reuse
coordinate-only factors instead of recomputing them at every grid point.

# Continue
### [Back to [top](#Table-of-Contents)]

- [Reference Metrics](../1-intro/reference_metric.ipynb)
- [Basis Transforms](basis_transforms.ipynb)
- [Curvilinear Wave Equation](../3-wave_equation/wave_equation_curvilinear.ipynb)